In [1]:
# ═══════════════════════════════════════════════════════════════════════
# MODULE 1 — DATA GENERATION (единственный источник миксов)
# v2: atomic savez + .done маркеры + robust cache validation
# ═══════════════════════════════════════════════════════════════════════
import os, json, random, zipfile, urllib.request, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

ROOT = Path("/content/drive/MyDrive/ESC50_Sound_Separation")
DATA_DIR = ROOT / "data" / "ESC-50"
AUDIO_DIR = DATA_DIR / "audio"
CACHE = ROOT / "cache"
CKPT = ROOT / "checkpoints"
OUT = ROOT / "outputs"
for d in [ROOT, CACHE, CKPT, OUT, CKPT/"sed", CKPT/"separation"]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Config
SR = 22050
DURATION = 4.0
N_SAMPLES = int(SR * DURATION)
N_FFT, HOP, N_MELS = 1024, 256, 128
N_FRAMES = 1 + N_SAMPLES // HOP
N_CLASSES = 50
S_MAX = 3
SRC_LEN_RANGE = (2.5, 3.5)
SRC_START_RANGE = (0.0, 0.5)
PEAK_NORM = 0.95
TRAIN_VARIANTS = 2
VAL_VARIANTS = 1

print(f"SR={SR}, N_SAMPLES={N_SAMPLES}, N_FRAMES={N_FRAMES}")

# ───────────────────────────────────────────────────────────────────────
# Atomic save helper
# ───────────────────────────────────────────────────────────────────────
def atomic_savez(path: Path, **arrays):
    """Сохраняет .npz атомарно + создаёт .done маркер."""
    path = Path(path)
    tmp = path.parent / (path.name + ".tmp")
    with open(tmp, "wb") as f:
        np.savez(f, **arrays)
    os.replace(tmp, path)
    marker = path.parent / (path.name + ".done")
    marker.touch()

# ───────────────────────────────────────────────────────────────────────
# Download ESC-50
# ───────────────────────────────────────────────────────────────────────
META_PATH = DATA_DIR / "meta" / "esc50.csv"
if not META_PATH.exists():
    ZIP = ROOT / "data" / "esc50.zip"
    ZIP.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP.exists():
        print("Downloading ESC-50...")
        urllib.request.urlretrieve(
            "https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip", ZIP)
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(ROOT / "data")
    (ROOT / "data" / "ESC-50-master").rename(DATA_DIR)

metadata = pd.read_csv(META_PATH).sort_values(["fold", "filename"]).reset_index(drop=True)
class_names = metadata[["target", "category"]].drop_duplicates().sort_values("target")["category"].tolist()
train_meta = metadata[metadata["fold"] != 5].reset_index(drop=True)
val_meta = metadata[metadata["fold"] == 5].reset_index(drop=True)
assert len(train_meta) == 1600 and len(val_meta) == 400

# ───────────────────────────────────────────────────────────────────────
# Audio cache (5s, 22050 Hz)
# ───────────────────────────────────────────────────────────────────────
def build_audio_cache(meta_df, cache_path):
    marker = cache_path.parent / (cache_path.name + ".done")
    if cache_path.exists() and marker.exists():
        return np.load(cache_path, mmap_mode='r')      # ← было без mmap

    arr = np.zeros((len(meta_df), 5 * SR), dtype=np.float32)
    for i, row in enumerate(tqdm(meta_df.itertuples(), desc=cache_path.name)):
        y, _ = librosa.load(AUDIO_DIR / row.filename, sr=SR, mono=True, duration=5.0)
        n = len(y)
        arr[i, :min(n, 5*SR)] = y[:5*SR]

    tmp = cache_path.parent / (cache_path.name + ".tmp.npy")
    np.save(tmp, arr)
    os.replace(tmp, cache_path)
    marker.touch()
    return np.load(cache_path, mmap_mode='r')          # ← и здесь

train_audio = build_audio_cache(train_meta, CACHE / "train_audio.npy")
val_audio = build_audio_cache(val_meta, CACHE / "val_audio.npy")
print(f"train_audio: {train_audio.shape}, val_audio: {val_audio.shape}")

# ───────────────────────────────────────────────────────────────────────
# Mixture generator (единственная реализация)
# ───────────────────────────────────────────────────────────────────────
def make_mixture(rng, meta_df, audio_cache):
    n_src = int(rng.integers(2, S_MAX + 1))
    classes = rng.choice(meta_df["target"].unique(), size=n_src, replace=False)

    source_meta = {"idx": [], "crop_start": [], "start": [], "length": [], "gain": []}
    class_ids = []
    source_buffers = np.zeros((n_src, N_SAMPLES), dtype=np.float32)

    for k, cid in enumerate(classes):
        candidates = meta_df[meta_df["target"] == cid]
        row = candidates.iloc[int(rng.integers(0, len(candidates)))]
        audio_idx = int(meta_df.index[meta_df["filename"] == row["filename"]][0])

        length = int(rng.uniform(*SRC_LEN_RANGE) * SR)
        max_crop = len(audio_cache[audio_idx]) - length
        crop_start = int(rng.integers(0, max_crop + 1))
        start = int(rng.uniform(*SRC_START_RANGE) * SR)
        gain = float(rng.uniform(0.5, 1.0))

        crop = audio_cache[audio_idx][crop_start:crop_start + length] * gain
        end = min(start + length, N_SAMPLES)
        source_buffers[k, start:end] = crop[:end - start]

        source_meta["idx"].append(audio_idx)
        source_meta["crop_start"].append(crop_start)
        source_meta["start"].append(start)
        source_meta["length"].append(length)
        source_meta["gain"].append(gain)
        class_ids.append(int(cid))

    mix_raw = source_buffers.sum(axis=0)
    peak = float(np.abs(mix_raw).max())
    scale = float(PEAK_NORM / peak) if peak > 1e-8 else 1.0
    mix = (mix_raw * scale).astype(np.float32)

    frame_labels = np.zeros((N_CLASSES, N_FRAMES), dtype=np.float32)
    clip_labels = np.zeros(N_CLASSES, dtype=np.float32)
    for cid, s, L in zip(class_ids, source_meta["start"], source_meta["length"]):
        f0 = s // HOP
        f1 = min(N_FRAMES, -(-(s + L) // HOP))
        frame_labels[cid, f0:f1] = 1.0
        clip_labels[cid] = 1.0

    return mix, frame_labels, clip_labels, source_meta, n_src, scale


def build_split(meta_df, audio_cache, variants, seed, split_name):
    N = len(meta_df) * variants
    mix = np.zeros((N, N_SAMPLES), dtype=np.float32)
    frame_labels = np.zeros((N, N_CLASSES, N_FRAMES), dtype=np.float32)
    clip_labels = np.zeros((N, N_CLASSES), dtype=np.float32)
    n_sources = np.zeros(N, dtype=np.int32)
    scale_arr = np.zeros(N, dtype=np.float32)
    src_idx = np.zeros((N, S_MAX), dtype=np.int32)
    src_crop = np.zeros((N, S_MAX), dtype=np.int32)
    src_start = np.zeros((N, S_MAX), dtype=np.int32)
    src_len = np.zeros((N, S_MAX), dtype=np.int32)
    src_gain = np.zeros((N, S_MAX), dtype=np.float32)

    out = 0
    for i in tqdm(range(len(meta_df)), desc=split_name):
        row_rng = np.random.default_rng(seed + i * 1009)
        for v in range(variants):
            mix_i, f_lab, c_lab, smeta, n_src, sc = make_mixture(row_rng, meta_df, audio_cache)
            mix[out] = mix_i; frame_labels[out] = f_lab; clip_labels[out] = c_lab
            n_sources[out] = n_src; scale_arr[out] = sc
            for k in range(n_src):
                src_idx[out, k] = smeta["idx"][k]
                src_crop[out, k] = smeta["crop_start"][k]
                src_start[out, k] = smeta["start"][k]
                src_len[out, k] = smeta["length"][k]
                src_gain[out, k] = smeta["gain"][k]
            out += 1
    return dict(mix=mix, frame_labels=frame_labels, clip_labels=clip_labels,
                n_sources=n_sources, scale=scale_arr,
                source_idx=src_idx, source_crop_start=src_crop,
                source_start=src_start, source_length=src_len, source_gain=src_gain)

# ───────────────────────────────────────────────────────────────────────
# Build or load (с .done валидацией)
# ───────────────────────────────────────────────────────────────────────
TRAIN_NPZ = CACHE / "train_mix.npz"
VAL_NPZ = CACHE / "val_mix.npz"
META_JSON = CACHE / "data_metadata.json"
TRAIN_DONE = CACHE / "train_mix.npz.done"
VAL_DONE = CACHE / "val_mix.npz.done"

EXPECTED_META = {
    "version": 2,
    "seed": SEED, "sr": SR, "n_samples": N_SAMPLES, "n_frames": N_FRAMES,
    "n_classes": N_CLASSES, "s_max": S_MAX,
    "train_variants": TRAIN_VARIANTS, "val_variants": VAL_VARIANTS,
    "src_len_range": list(SRC_LEN_RANGE), "src_start_range": list(SRC_START_RANGE),
    "peak_norm": PEAK_NORM,
}

def cache_valid():
    if not (TRAIN_NPZ.exists() and VAL_NPZ.exists() and META_JSON.exists()):
        return False
    if not (TRAIN_DONE.exists() and VAL_DONE.exists()):
        return False
    try:
        saved = json.load(open(META_JSON))
        if saved != EXPECTED_META:
            return False
        # Sanity: проверяем форму первого ключа
        with np.load(TRAIN_NPZ) as z:
            assert z['mix'].shape == (len(train_meta) * TRAIN_VARIANTS, N_SAMPLES)
        return True
    except Exception as e:
        print(f"Cache invalid: {e}")
        return False

if not cache_valid():
    print("Generating mixtures (this may take ~15 min)...")
    train_data = build_split(train_meta, train_audio, TRAIN_VARIANTS, SEED, "train")
    atomic_savez(TRAIN_NPZ, **train_data)
    val_data = build_split(val_meta, val_audio, VAL_VARIANTS, SEED + 100000, "val")
    atomic_savez(VAL_NPZ, **val_data)
    json.dump(EXPECTED_META, open(META_JSON, "w"), indent=2)
    print("Cache saved.")
else:
    print("Loading existing cache...")

# Memmap load
train_npz = np.load(TRAIN_NPZ, mmap_mode='r')
val_npz = np.load(VAL_NPZ, mmap_mode='r')

# Sanity
assert train_npz['mix'].shape == (len(train_meta) * TRAIN_VARIANTS, N_SAMPLES)
assert val_npz['mix'].shape == (len(val_meta) * VAL_VARIANTS, N_SAMPLES)
assert np.all(train_npz['clip_labels'].sum(axis=1) >= 2)
assert np.all(val_npz['clip_labels'].sum(axis=1) >= 2)
assert np.all(train_npz['frame_labels'].max(axis=2) <= train_npz['clip_labels'])
print(f"Train: {train_npz['mix'].shape}, Val: {val_npz['mix'].shape}")

# Reconstruction helper
def reconstruct_sources(npz_data, audio_cache, idx, s_max=S_MAX):
    n = int(npz_data['n_sources'][idx])
    scale = float(npz_data['scale'][idx])
    sources = np.zeros((s_max, N_SAMPLES), dtype=np.float32)
    for k in range(n):
        a_idx = int(npz_data['source_idx'][idx, k])
        c_start = int(npz_data['source_crop_start'][idx, k])
        s_start = int(npz_data['source_start'][idx, k])
        s_len = int(npz_data['source_length'][idx, k])
        gain = float(npz_data['source_gain'][idx, k])
        crop = audio_cache[a_idx][c_start:c_start + s_len] * gain * scale
        end = min(s_start + s_len, N_SAMPLES)
        sources[k, s_start:end] = crop[:end - s_start]
    return sources, n

# Verify reconstruction == mix
for test_idx in [0, 100, 1000, -1]:
    src, _ = reconstruct_sources(train_npz, train_audio, test_idx)
    err = np.abs(src.sum(0) - train_npz['mix'][test_idx]).max()
    assert err < 1e-5, f"Reconstruction error at idx {test_idx}: {err}"
print("Reconstruction check passed.")

print(f"\nModule 1 done. Train: {len(train_npz['mix'])}, Val: {len(val_npz['mix'])}")

Mounted at /content/drive
SR=22050, N_SAMPLES=88200, N_FRAMES=345
train_audio: (1600, 110250), val_audio: (400, 110250)
Loading existing cache...
Train: (3200, 88200), Val: (400, 88200)
Reconstruction check passed.

Module 1 done. Train: 3200, Val: 400


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# MODULE 2 — SED (2048-dim hook + регуляризация + SpecAugment)
# v4
# ═══════════════════════════════════════════════════════════════════════
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy.ndimage import median_filter, maximum_filter1d
from scipy.optimize import linear_sum_assignment
import time

try:
    from panns_inference import SoundEventDetection
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "panns-inference"])
    from panns_inference import SoundEventDetection

# ═══════════════════════════════════════════════════════════════════════
# 2.0 BOOTSTRAP — самодостаточная загрузка из Drive
# ═══════════════════════════════════════════════════════════════════════
import os, json, random, warnings
from pathlib import Path
import numpy as np
import torch
import pandas as pd
warnings.filterwarnings("ignore")

try:
    # Проверяем, всё ли на месте (NameError при первом отсутствии)
    _ = (train_data, val_data, train_audio, val_audio,
         N_SAMPLES, N_FFT, HOP, S_MAX, CACHE, CKPT, DEVICE,
         atomic_torch_save, reconstruct_sources)
    print("Bootstrap: не нужен — зависимости уже в памяти.")
except NameError as e:
    print(f"Авто-bootstrap: '{e.name}' не найден. Загружаю из Drive (~15 сек)...")

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # ─── Пути ───
    ROOT     = Path("/content/drive/MyDrive/ESC50_Sound_Separation")
    DATA_DIR = ROOT / "data" / "ESC-50"
    CACHE    = ROOT / "cache"
    CKPT     = ROOT / "checkpoints"
    OUT      = ROOT / "outputs"

    # ─── Константы ───
    SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    SR        = 22050
    DURATION  = 4.0
    N_SAMPLES = int(SR * DURATION)          # 88200
    N_FFT     = 1024
    HOP       = 256
    N_MELS    = 128
    N_FRAMES  = 1 + N_SAMPLES // HOP        # 345
    N_CLASSES = 50
    S_MAX     = 3
    F_BINS    = N_FFT // 2 + 1              # 513

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  DEVICE = {DEVICE}")

    # ─── atomic save ───
    def atomic_torch_save(state, path):
        path = Path(path)
        tmp = path.parent / (path.name + ".tmp")
        torch.save(state, tmp)
        os.replace(tmp, path)

    # ─── Метаданные ───
    metadata = pd.read_csv(DATA_DIR / "meta" / "esc50.csv").sort_values(
        ["fold", "filename"]).reset_index(drop=True)
    class_names = metadata[["target", "category"]].drop_duplicates(
    ).sort_values("target")["category"].tolist()
    train_meta = metadata[metadata["fold"] != 5].reset_index(drop=True)
    val_meta   = metadata[metadata["fold"] == 5].reset_index(drop=True)

    # ─── Данные ───
    def _load_split(split):
        d = CACHE / f"{split}_mix_npy"
        if not d.exists():
            raise FileNotFoundError(
                f"{d} не найден. Запустите Module 1 (или Bootstrap).")
        keys = ["mix", "frame_labels", "clip_labels", "n_sources", "scale",
                "source_idx", "source_crop_start", "source_start",
                "source_length", "source_gain"]
        return {k: np.load(d / f"{k}.npy", mmap_mode='r') for k in keys}

    train_data  = _load_split("train")
    val_data    = _load_split("val")
    train_audio = np.load(CACHE / "train_audio.npy", mmap_mode='r')
    val_audio   = np.load(CACHE / "val_audio.npy",   mmap_mode='r')

    # ─── reconstruct_sources ───
    def reconstruct_sources(data_dict, audio_cache, idx, s_max=S_MAX):
        n = int(data_dict['n_sources'][idx])
        scale = float(data_dict['scale'][idx])
        sources = np.zeros((s_max, N_SAMPLES), dtype=np.float32)
        for k in range(n):
            a_idx = int(data_dict['source_idx'][idx, k])
            c_start = int(data_dict['source_crop_start'][idx, k])
            s_start = int(data_dict['source_start'][idx, k])
            s_len = int(data_dict['source_length'][idx, k])
            gain = float(data_dict['source_gain'][idx, k])
            crop = audio_cache[a_idx][c_start:c_start + s_len] * gain * scale
            end = min(s_start + s_len, N_SAMPLES)
            sources[k, s_start:end] = crop[:end - s_start]
        return sources, n

    print(f"  train={train_data['mix'].shape[0]}, "
          f"val={val_data['mix'].shape[0]}, "
          f"audio_train={train_audio.shape[0]}")
    print("✅ Авто-bootstrap завершён.")

# ═══════════════════════════════════════════════════════════════════════
# DEVICE + флаги (мягкая проверка для CPU-режима)
# ═══════════════════════════════════════════════════════════════════════
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Флаги явного разрешения
DO_TRAIN         = False    # ← True только при наличии GPU
ALLOW_CPU_TRAIN  = False    # ← True если готовы ждать 15+ часов на CPU
ALLOW_CPU_EXTRACT = False   # ← True если хотите извлечь признаки на CPU (~3 ч)

# Проверяем, что признаки уже на Drive
_feat_dir = CACHE / "panns_feats_2048"
_features_ready = (
    (_feat_dir / "train.npy").exists() and (_feat_dir / "train.npy.done").exists()
    and (_feat_dir / "val.npy").exists() and (_feat_dir / "val.npy.done").exists()
)
_ckpt_ready = (CKPT / "sed" / "best_2048_v4.pt").exists()

if DEVICE.type != "cuda":
    print("=" * 65)
    print("⚠ GPU недоступен — режим CPU")
    print("=" * 65)
    print(f"  Признаки извлечены: {'✅' if _features_ready else '❌'}")
    print(f"  Best чекпоинт есть: {'✅' if _ckpt_ready else '❌'}")
    print(f"  DO_TRAIN = {DO_TRAIN}")
    print(f"  ALLOW_CPU_TRAIN = {ALLOW_CPU_TRAIN}")
    print(f"  ALLOW_CPU_EXTRACT = {ALLOW_CPU_EXTRACT}")
    print()

    if not _features_ready and not ALLOW_CPU_EXTRACT:
        raise RuntimeError(
            "GPU нет, признаки не извлечены, извлечение на CPU запрещено.\n"
            "Варианты:\n"
            "  1) Дождаться GPU и перезапустить\n"
            "  2) Перейти на Kaggle (30 ч/неделю T4)\n"
            "  3) ALLOW_CPU_EXTRACT = True (медленно, ~3 ч)"
        )
    if DO_TRAIN and not ALLOW_CPU_TRAIN:
        raise RuntimeError(
            "DO_TRAIN=True, но ALLOW_CPU_TRAIN=False.\n"
            "Обучение на CPU займёт 15+ часов.\n"
            "Установите ALLOW_CPU_TRAIN=True, чтобы действительно запустить."
        )
    if not DO_TRAIN and _features_ready and _ckpt_ready:
        print("→ Режим 'load only': признаки и чекпоинт загружены, обучение пропускается.")
        print("  Полезно для проверки состояния и подготовки Module 3.")
    print("=" * 65)

# ─── Config (Path A регуляризация) ───
SED_CFG = dict(
    panns_sr=32000, panns_batch=8, head_batch=64,
    feat_dim=2048,
    hidden=128,                # было 256 — компактнее
    layers=1,                  # было 2
    dropout=0.5,               # было 0.3
    epochs=50,                 # было 70
    patience=8,                # было 12
    lr=5e-4,                   # было 1e-3
    wd=1e-3,                   # было 1e-4 (10×)
    pos_cap=5.0, lambda_clip=1.0, smooth=5,
    spec_aug_time_mask=30,     # макс. длина маски по времени (кадров)
    spec_aug_n_masks=2,        # сколько масок на пример
    val_loss_rise_tol=1.02,    # рост val_loss на 2% → +1 к счётчику
    val_loss_rise_max=5,       # 5 подряд → early stop
)
SED_VERSION = 4

# ─── Atomic save ───
def atomic_torch_save(state, path):
    path = Path(path)
    tmp = path.parent / (path.name + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)

# ─── npz → npy конвертер (для memmap) ───
def npz_to_npy(npz_path):
    npz_path = Path(npz_path)
    out_dir = npz_path.parent / (npz_path.stem + "_npy")
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"Converting {npz_path.name} → {out_dir.name}/")
    with np.load(npz_path) as z:
        for key in z.files:
            npy_path = out_dir / f"{key}.npy"
            done = out_dir / f"{key}.done"
            if npy_path.exists() and done.exists():
                print(f"  {key}: already")
                continue
            print(f"  {key}: {z[key].nbytes/1e9:.2f} GB ...", end=" ", flush=True)
            tmp = out_dir / f"{key}.tmp.npy"
            np.save(tmp, z[key])
            os.replace(tmp, npy_path)
            done.touch()
            print("done")

    keys = np.load(npz_path).files
    return {k: np.load(out_dir / f"{k}.npy", mmap_mode='r') for k in keys}

train_data = npz_to_npy(TRAIN_NPZ)
val_data = npz_to_npy(VAL_NPZ)
print(f"train_data['mix']: {train_data['mix'].shape} {train_data['mix'].dtype}")

# ─── PANNs checkpoint ───
PANNS_CKPT = ROOT / "panns_data" / "Cnn14_DecisionLevelMax.pth"
PANNS_CSV = ROOT / "panns_data" / "class_labels_indices.csv"
PANNS_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not PANNS_CSV.exists():
    urllib.request.urlretrieve(
        "http://storage.googleapis.com/us_audioset/youtube_corpus/v1/csv/class_labels_indices.csv",
        str(PANNS_CSV))
if not PANNS_CKPT.exists():
    print("Downloading PANNs (~327 MB)...")
    urllib.request.urlretrieve(
        "https://zenodo.org/record/3987831/files/Cnn14_DecisionLevelMax_mAP%3D0.385.pth?download=1",
        str(PANNS_CKPT))

# ─── Hook ───
def find_target_layer(model):
    candidates = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.out_channels == 2048:
            candidates.append((name, module))
    if candidates:
        return candidates[-1]
    for name, module in model.named_modules():
        if 'conv_block6' in name.lower():
            return name, module
    raise RuntimeError("Слой с 2048 каналами не найден")

# ─── Feature extraction (memmap, .cpu(), periodic flush) ───
def extract_features_with_hook(mix_data, out_path, split_name):
    marker = out_path.parent / (out_path.name + ".done")
    n = len(mix_data)
    expected_size = n * N_FRAMES * SED_CFG['feat_dim'] * 2

    if out_path.exists() and marker.exists() and out_path.stat().st_size == expected_size:
        print(f"{split_name}: already extracted")
        return

    print(f"Extracting {split_name} ({n} samples)...")
    panns = SoundEventDetection(checkpoint_path=str(PANNS_CKPT), device=str(DEVICE))
    panns.model.eval()
    target_name, target_module = find_target_layer(panns.model)
    print(f"Hook: {target_name}")

    captured = {}
    def hook_fn(m, i, o): captured['feat'] = o
    handle = target_module.register_forward_hook(hook_fn)

    tmp = out_path.parent / (out_path.name + ".tmp")
    feats = np.lib.format.open_memmap(str(tmp), mode='w+', dtype=np.float16,
                                       shape=(n, N_FRAMES, SED_CFG['feat_dim']))
    try:
        for i, start in enumerate(tqdm(range(0, n, SED_CFG['panns_batch']),
                                        desc=split_name)):
            end = min(start + SED_CFG['panns_batch'], n)
            batch = mix_data[start:end]
            prepared = []
            for a in batch:
                a32 = librosa.resample(a, orig_sr=SR, target_sr=32000)
                L = int(DURATION * 32000)
                a32 = np.pad(a32, (0, max(0, L - len(a32))))[:L]
                prepared.append(a32.astype(np.float32))

            with torch.inference_mode():
                _ = panns.inference(np.stack(prepared))

            feat = captured['feat']                        # (B, 2048, F', T')
            feat = feat.mean(dim=2)                        # (B, 2048, T')
            feat = F.interpolate(feat, size=N_FRAMES,
                                 mode='linear', align_corners=False)
            feat = feat.transpose(1, 2).cpu().numpy().astype(np.float16)
            feats[start:end] = feat

            del captured['feat'], feat
            if i % 20 == 0:
                feats.flush()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        feats.flush()
    finally:
        handle.remove()
        del feats, panns
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    os.replace(tmp, out_path)
    marker.touch()
    print(f"{split_name}: saved ({out_path.stat().st_size/1e9:.2f} GB)")

FEAT_DIR = CACHE / "panns_feats_2048"
FEAT_DIR.mkdir(exist_ok=True)
FEAT_TRAIN = FEAT_DIR / "train.npy"
FEAT_VAL = FEAT_DIR / "val.npy"

# Extraction — только если нужно и разрешено
if not _features_ready:
    if DEVICE.type != "cuda" and not ALLOW_CPU_EXTRACT:
        raise RuntimeError("Признаки не извлечены, GPU нет, ALLOW_CPU_EXTRACT=False")
    extract_features_with_hook(train_data['mix'], FEAT_TRAIN, "train")
    extract_features_with_hook(val_data['mix'],   FEAT_VAL,   "val")
else:
    print("Features already extracted — skip extraction")

X_train = np.load(FEAT_TRAIN, mmap_mode='r')
X_val = np.load(FEAT_VAL, mmap_mode='r')
print(f"Features: train {X_train.shape}, val {X_val.shape}")

# ─── Model (компактнее) ───
class BiGRUHead(nn.Module):
    def __init__(self, in_dim=2048, num_classes=N_CLASSES,
                 hidden=128, layers=1, dropout=0.5):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.gru = nn.GRU(hidden, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True, dropout=dropout if layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.frame_head = nn.Linear(hidden * 2, num_classes)
        self.clip_head = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes))

    def forward(self, x):
        x = self.proj(x)
        h, _ = self.gru(x)
        h = self.dropout(h)
        return self.frame_head(h).transpose(1, 2), self.clip_head(h.amax(dim=1))

# ─── SpecAugment (только train) ───
def spec_augment(x, time_mask_max, n_masks):
    """x: (B, T, F). Маскируем случайные отрезки по времени нулями."""
    B, T, _ = x.shape
    for b in range(B):
        for _ in range(n_masks):
            t = int(torch.randint(1, time_mask_max + 1, (1,)).item())
            t0 = int(torch.randint(0, max(1, T - t), (1,)).item())
            x[b, t0:t0+t, :] = 0.0
    return x

# ─── pos_weight ───
def compute_pos_weights(frame_labels, clip_labels, sample=500, cap=5.0):
    N = len(frame_labels)
    idx = np.random.RandomState(0).choice(N, min(sample, N), replace=False)
    yf = frame_labels[np.sort(idx)].astype(np.float32)
    yc = clip_labels[np.sort(idx)].astype(np.float32)
    fp = yf.sum(axis=(0, 2)); cp = yc.sum(axis=0)
    n, T = yf.shape[0], yf.shape[2]
    f_pw = np.clip((n * T - fp) / (fp + 1e-6), 1, cap)
    c_pw = np.clip((n - cp) / (cp + 1e-6), 1, cap)
    return (torch.tensor(f_pw).float().view(1, -1, 1),
            torch.tensor(c_pw).float())

# ─── Metrics ───
def _smooth(prob, kernel=5):
    if kernel <= 1: return prob
    size = (1,) * (prob.ndim - 1) + (kernel,)
    return median_filter(prob, size=size, mode='nearest')

def _events(b):
    p = np.concatenate([[0], b.astype(np.int8), [0]])
    d = np.diff(p)
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)

def event_f1_from_prob(prob, target, thr=0.5, iou_thr=0.5):
    pred = prob > thr; tgt = target > 0.5
    tp = fp = fn = 0
    for b in range(pred.shape[0]):
        for k in range(pred.shape[1]):
            ps, pe = _events(pred[b, k])
            ts, te = _events(tgt[b, k])
            if len(ts) == 0: fp += len(ps); continue
            if len(ps) == 0: fn += len(ts); continue
            matched = np.zeros(len(ts), dtype=bool)
            for i in range(len(ps)):
                bi, bj = 0.0, -1
                for j in range(len(ts)):
                    if matched[j]: continue
                    inter = max(0, min(pe[i], te[j]) - max(ps[i], ts[j]))
                    union = max(pe[i], te[j]) - min(ps[i], ts[j])
                    iou = inter / (union + 1e-8)
                    if iou > bi: bi, bj = iou, j
                if bi >= iou_thr: matched[bj] = True; tp += 1
                else: fp += 1
            fn += int((~matched).sum())
    p = tp / (tp + fp + 1e-8); r = tp / (tp + fn + 1e-8)
    return 2*p*r / (p + r + 1e-8), p, r

def frame_f1(prob, target, thr=0.5, tol=2):
    pred = prob > thr; tgt = target > 0.5
    if tol > 0:
        tgt = maximum_filter1d(tgt.astype(np.float32), size=2*tol+1, axis=-1) > 0.5
    tp = (pred & tgt).sum(); fp = (pred & ~tgt).sum(); fn = (~pred & tgt).sum()
    return float(2*tp / (2*tp + fp + fn + 1e-8))

def clip_map_from_logits(clip_logits, clip_labels):
    prob = torch.sigmoid(clip_logits).cpu().numpy()
    y = clip_labels.cpu().numpy()
    aps = []
    for k in range(y.shape[1]):
        if y[:, k].sum() == 0: continue
        order = np.argsort(-prob[:, k]); ys = y[order, k]
        tp_cum = np.cumsum(ys)
        prec = tp_cum / (np.arange(len(ys)) + 1)
        aps.append((prec * ys).sum() / ys.sum())
    return float(np.mean(aps)) if aps else 0.0

# ─── Train/eval ───
def train_epoch(model, X, Yf, Yc, opt, pw_f, pw_c, batch=64):
    model.train()
    N = len(X); order = np.random.permutation(N)
    total, n = 0.0, 0
    for i in tqdm(range(0, N, batch), desc="train", leave=False):
        b = np.sort(order[i:i+batch])
        xb = torch.from_numpy(X[b].astype(np.float32)).to(DEVICE)
        ft = torch.from_numpy(np.asarray(Yf[b])).to(DEVICE)
        ct = torch.from_numpy(np.asarray(Yc[b])).to(DEVICE)

        # SpecAugment только в train
        xb = spec_augment(xb, SED_CFG['spec_aug_time_mask'], SED_CFG['spec_aug_n_masks'])

        fl, cl = model(xb)
        loss = (F.binary_cross_entropy_with_logits(fl, ft, pos_weight=pw_f)
                + SED_CFG['lambda_clip'] * F.binary_cross_entropy_with_logits(cl, ct, pos_weight=pw_c))
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        total += loss.item(); n += 1
    return total / max(1, n)

@torch.no_grad()
def evaluate(model, X, Yf, Yc, pw_f, pw_c, batch=256):
    model.eval()
    fl_all, cl_all = [], []
    tot, n = 0.0, 0
    for i in range(0, len(X), batch):
        b = np.arange(i, min(i + batch, len(X)))
        xb = torch.from_numpy(X[b].astype(np.float32)).to(DEVICE)
        ft = torch.from_numpy(np.asarray(Yf[b])).to(DEVICE)
        ct = torch.from_numpy(np.asarray(Yc[b])).to(DEVICE)
        fl, cl = model(xb)
        tot += (F.binary_cross_entropy_with_logits(fl, ft, pos_weight=pw_f)
                + SED_CFG['lambda_clip'] * F.binary_cross_entropy_with_logits(cl, ct, pos_weight=pw_c)).item()
        n += 1
        fl_all.append(fl.cpu()); cl_all.append(cl.cpu())

    fl_all = torch.cat(fl_all); cl_all = torch.cat(cl_all)
    prob = _smooth(torch.sigmoid(fl_all).numpy(), SED_CFG['smooth'])
    Yf_np = np.asarray(Yf); Yc_t = torch.from_numpy(np.asarray(Yc)).float()

    f_f1 = frame_f1(prob, Yf_np)
    ev, ep, er = event_f1_from_prob(prob, Yf_np, thr=0.5)
    best_f1, best_thr = 0.0, 0.5
    for t in np.arange(0.1, 0.91, 0.1):
        f1, _, _ = event_f1_from_prob(prob, Yf_np, thr=float(t))
        if f1 > best_f1: best_f1, best_thr = f1, float(t)
    mAP = clip_map_from_logits(cl_all, Yc_t)
    return dict(loss=tot/n, frame_f1=f_f1, event_f1=ev, event_p=ep, event_r=er,
                event_f1_tuned=best_f1, event_thr=best_thr, clip_map=mAP)

# ═══════════════════════════════════════════════════════════════════════
# Training loop (resume + hard stop по val_loss_rising)
# ═══════════════════════════════════════════════════════════════════════
RESUME = True   # ← True для продолжения

CKPT_SED = CKPT / "sed" / "best_2048_v4.pt"
CKPT_SED_LAST = CKPT / "sed" / "last_2048_v4.pt"

model = BiGRUHead(in_dim=SED_CFG['feat_dim'], hidden=SED_CFG['hidden'],
                  layers=SED_CFG['layers'], dropout=SED_CFG['dropout']).to(DEVICE)
print(f"BiGRUHead params: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

opt = torch.optim.AdamW(model.parameters(), lr=SED_CFG['lr'], weight_decay=SED_CFG['wd'])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30, eta_min=1e-5)

pw_f, pw_c = compute_pos_weights(train_data['frame_labels'], train_data['clip_labels'],
                                  sample=500, cap=SED_CFG['pos_cap'])
pw_f, pw_c = pw_f.to(DEVICE), pw_c.to(DEVICE)
print(f"pos_weight: frame={pw_f.median():.2f}, clip={pw_c.median():.2f}")

start_epoch = 1
best_score = -1.0
best_state = None
wait = 0
val_loss_rising = 0
best_val_loss = float('inf')
history = []

if RESUME and CKPT_SED_LAST.exists():
    print("Loading last checkpoint...")
    last = torch.load(CKPT_SED_LAST, map_location=DEVICE, weights_only=False)
    if last.get('version') != SED_VERSION:
        raise RuntimeError(f"Version mismatch: {last.get('version')} != {SED_VERSION}")
    model.load_state_dict(last['model_state'])
    opt.load_state_dict(last['optimizer_state'])
    sched.load_state_dict(last['scheduler_state'])
    start_epoch = last['epoch'] + 1
    best_score = last['best_score']
    wait = last['wait']
    val_loss_rising = last.get('val_loss_rising', 0)
    best_val_loss = last.get('best_val_loss', float('inf'))
    history = last['history']
    if CKPT_SED.exists():
        best_state = torch.load(CKPT_SED, map_location=DEVICE, weights_only=False)['model_state']
    print(f"Resumed from E{start_epoch}, best={best_score:.4f}")

if DO_TRAIN:
    for epoch in range(start_epoch, SED_CFG['epochs'] + 1):
        t0 = time.time()
        tr = train_epoch(model, X_train, train_data['frame_labels'],
                         train_data['clip_labels'], opt, pw_f, pw_c,
                         batch=SED_CFG['head_batch'])
        va = evaluate(model, X_val, val_data['frame_labels'],
                      val_data['clip_labels'], pw_f, pw_c)
        sched.step()
        # ... (остальной код цикла без изменений) ...
        history.append(dict(epoch=epoch, train_loss=tr, **va))
        print(f"E{epoch:02d} | tr={tr:.3f} val={va['loss']:.3f} | "
              f"fF1={va['frame_f1']:.3f} | evF1={va['event_f1']:.3f} "
              f"(tuned={va['event_f1_tuned']:.3f}@{va['event_thr']:.2f}) | "
              f"mAP={va['clip_map']:.3f} | {time.time()-t0:.0f}s")

        # счётчик роста val_loss
        if va['loss'] > best_val_loss * SED_CFG['val_loss_rise_tol']:
            val_loss_rising += 1
        else:
            best_val_loss = va['loss']
            val_loss_rising = 0

        # best по event_f1_tuned
        if va['event_f1_tuned'] > best_score:
            best_score = va['event_f1_tuned']
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
            atomic_torch_save(
                dict(version=SED_VERSION, model_state=best_state, config=SED_CFG,
                    history=history, best_event_f1_tuned=best_score,
                    class_names=class_names, epoch=epoch),
                CKPT_SED)
            print(f"  → best {best_score:.4f}")
        else:
            wait += 1

        atomic_torch_save(
            dict(version=SED_VERSION,
                model_state=model.state_dict(),
                optimizer_state=opt.state_dict(),
                scheduler_state=sched.state_dict(),
                epoch=epoch, best_score=best_score, wait=wait,
                val_loss_rising=val_loss_rising, best_val_loss=best_val_loss,
                history=history),
            CKPT_SED_LAST)

        if wait >= SED_CFG['patience'] or val_loss_rising >= SED_CFG['val_loss_rise_max']:
            print(f"Early stop E{epoch} (wait={wait}, val_rise={val_loss_rising})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"\nSED training done. Best tuned event F1: {best_score:.4f}")
    print(f"Best val loss: {best_val_loss:.4f}")
else:
    print("\n" + "=" * 65)
    print("DO_TRAIN=False — обучение пропущено")
    print("=" * 65)
    if _ckpt_ready:
        print("Загружаю best_2048_v4.pt для инференса...")
        best = torch.load(CKPT_SED, map_location=DEVICE, weights_only=False)
        model.load_state_dict(best['model_state'])
        model.eval()
        best_score = best['best_event_f1_tuned']
        print(f"  epoch={best['epoch']}, "
              f"event_f1_tuned={best_score:.4f}, "
              f"config: hidden={best['config']['hidden']}, "
              f"layers={best['config']['layers']}")
        print("  Модель готова к инференсу.")
    else:
        print("⚠ best_2048_v4.pt не найден — модель в случайном состоянии.")
        print("  Для инференса нужен GPU + обучение, либо копия чекпоинта с Drive.")

Авто-bootstrap: 'train_data' не найден. Загружаю из Drive (~15 сек)...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  DEVICE = cuda
  train=3200, val=400, audio_train=1600
✅ Авто-bootstrap завершён.
Converting train_mix.npz → train_mix_npy/
  mix: already
  frame_labels: already
  clip_labels: already
  n_sources: already
  scale: already
  source_idx: already
  source_crop_start: already
  source_start: already
  source_length: already
  source_gain: already
Converting val_mix.npz → val_mix_npy/
  mix: already
  frame_labels: already
  clip_labels: already
  n_sources: already
  scale: already
  source_idx: already
  source_crop_start: already
  source_start: already
  source_length: already
  source_gain: already
train_data['mix']: (3200, 88200) float32
Features already extracted — skip extraction
Features: train (3200, 345, 2048), val (400, 345, 2048)
BiGRUHead params: 0.51M
pos_weight: frame=5.00, c

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# MODULE 3 — SOURCE SEPARATION
# v4: magnitude_sdr (softmax + L1/SDR) | complex (cIRM)
# ═══════════════════════════════════════════════════════════════════════
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import linear_sum_assignment
import torch, torch.nn as nn, torch.nn.functional as F
import time, os

# ═══════════════════════════════════════════════════════════════════════
# 3.0 BOOTSTRAP — самодостаточная загрузка из Drive
# ═══════════════════════════════════════════════════════════════════════
import os, json, random, warnings
from pathlib import Path
import numpy as np
import torch
import pandas as pd
warnings.filterwarnings("ignore")

try:
    # Проверяем, всё ли на месте (NameError при первом отсутствии)
    _ = (train_data, val_data, train_audio, val_audio,
         N_SAMPLES, N_FFT, HOP, S_MAX, CACHE, CKPT, DEVICE,
         atomic_torch_save, reconstruct_sources)
    print("Bootstrap: не нужен — зависимости уже в памяти.")
except NameError as e:
    print(f"Авто-bootstrap: '{e.name}' не найден. Загружаю из Drive (~15 сек)...")

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # ─── Пути ───
    ROOT     = Path("/content/drive/MyDrive/ESC50_Sound_Separation")
    DATA_DIR = ROOT / "data" / "ESC-50"
    CACHE    = ROOT / "cache"
    CKPT     = ROOT / "checkpoints"
    OUT      = ROOT / "outputs"

    # ─── Константы ───
    SEED = 42
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

    SR        = 22050
    DURATION  = 4.0
    N_SAMPLES = int(SR * DURATION)          # 88200
    N_FFT     = 1024
    HOP       = 256
    N_MELS    = 128
    N_FRAMES  = 1 + N_SAMPLES // HOP        # 345
    N_CLASSES = 50
    S_MAX     = 3
    F_BINS    = N_FFT // 2 + 1              # 513

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  DEVICE = {DEVICE}")

    # ─── atomic save ───
    def atomic_torch_save(state, path):
        path = Path(path)
        tmp = path.parent / (path.name + ".tmp")
        torch.save(state, tmp)
        os.replace(tmp, path)

    # ─── Метаданные ───
    metadata = pd.read_csv(DATA_DIR / "meta" / "esc50.csv").sort_values(
        ["fold", "filename"]).reset_index(drop=True)
    class_names = metadata[["target", "category"]].drop_duplicates(
    ).sort_values("target")["category"].tolist()
    train_meta = metadata[metadata["fold"] != 5].reset_index(drop=True)
    val_meta   = metadata[metadata["fold"] == 5].reset_index(drop=True)

    # ─── Данные ───
    def _load_split(split):
        d = CACHE / f"{split}_mix_npy"
        if not d.exists():
            raise FileNotFoundError(
                f"{d} не найден. Запустите Module 1 (или Bootstrap).")
        keys = ["mix", "frame_labels", "clip_labels", "n_sources", "scale",
                "source_idx", "source_crop_start", "source_start",
                "source_length", "source_gain"]
        return {k: np.load(d / f"{k}.npy", mmap_mode='r') for k in keys}

    train_data  = _load_split("train")
    val_data    = _load_split("val")
    train_audio = np.load(CACHE / "train_audio.npy", mmap_mode='r')
    val_audio   = np.load(CACHE / "val_audio.npy",   mmap_mode='r')

    # ─── reconstruct_sources ───
    def reconstruct_sources(data_dict, audio_cache, idx, s_max=S_MAX):
        n = int(data_dict['n_sources'][idx])
        scale = float(data_dict['scale'][idx])
        sources = np.zeros((s_max, N_SAMPLES), dtype=np.float32)
        for k in range(n):
            a_idx = int(data_dict['source_idx'][idx, k])
            c_start = int(data_dict['source_crop_start'][idx, k])
            s_start = int(data_dict['source_start'][idx, k])
            s_len = int(data_dict['source_length'][idx, k])
            gain = float(data_dict['source_gain'][idx, k])
            crop = audio_cache[a_idx][c_start:c_start + s_len] * gain * scale
            end = min(s_start + s_len, N_SAMPLES)
            sources[k, s_start:end] = crop[:end - s_start]
        return sources, n

    print(f"  train={train_data['mix'].shape[0]}, "
          f"val={val_data['mix'].shape[0]}, "
          f"audio_train={train_audio.shape[0]}")
    print("✅ Авто-bootstrap завершён.")

# ═══════════════════════════════════════════════════════════════════════
# 3.1 Конфигурация
# ═══════════════════════════════════════════════════════════════════════
MODE = "magnitude_sdr"    # ← "magnitude_sdr" | "complex"

# Флаги для CPU-режима (как в Module 2)
DO_TRAIN         = True    # ← True когда есть GPU
ALLOW_CPU_TRAIN  = False    # ← никогда не ставьте True без причины

SEP_VERSION = 4
SEP_CFG = dict(
    batch_size=8,
    base=32,
    epochs=25,
    lr=3e-4 if MODE == "magnitude_sdr" else 1e-4,
    wd=1e-5,
    patience=6,
    gradient_clip=5.0,
    sdr_loss_weight=0.5,        # только magnitude_sdr
    unmatched_loss_weight=0.3,
)

print(f"MODE = {MODE}")
print(f"SEP_CFG = {SEP_CFG}")

if DEVICE.type != "cuda":
    print("\n" + "=" * 65)
    print("⚠ GPU недоступен — режим CPU")
    print("=" * 65)
    print(f"  DO_TRAIN = {DO_TRAIN}")
    print(f"  ALLOW_CPU_TRAIN = {ALLOW_CPU_TRAIN}")
    if DO_TRAIN and not ALLOW_CPU_TRAIN:
        raise RuntimeError(
            "DO_TRAIN=True, но ALLOW_CPU_TRAIN=False.\n"
            "Separation на CPU займёт 40+ часов. Установите ALLOW_CPU_TRAIN=True "
            "только если действительно готовы ждать."
        )
    if not DO_TRAIN:
        print("\n→ Режим 'build only': модель построена, обучение пропущено.")
        print("  Данные и код проверены, готовы к запуску на GPU.")
    print("=" * 65)

# ═══════════════════════════════════════════════════════════════════════
# 3.2 STFT / ISTFT
# ═══════════════════════════════════════════════════════════════════════
_HANN = torch.hann_window(N_FFT).to(DEVICE)

def stft(x):
    return torch.stft(x, n_fft=N_FFT, hop_length=HOP, window=_HANN,
                      center=True, return_complex=True)

def istft(spec, length=N_SAMPLES):
    return torch.istft(spec, n_fft=N_FFT, hop_length=HOP, window=_HANN,
                       center=True, length=length)

# ═══════════════════════════════════════════════════════════════════════
# 3.3 Dataset
# ═══════════════════════════════════════════════════════════════════════
class SepDataset(Dataset):
    """Оборачивает memmap-данные из Module 1."""
    def __init__(self, data_dict, audio_cache):
        self.data = data_dict
        self.audio = audio_cache

    def __len__(self):
        return len(self.data['mix'])

    def __getitem__(self, idx):
        mix = torch.from_numpy(np.asarray(self.data['mix'][idx]))
        src, n = reconstruct_sources(self.data, self.audio, idx)
        return dict(
            mix=mix,
            sources=torch.from_numpy(src),
            n_sources=torch.tensor(n, dtype=torch.long),
        )

train_ds = SepDataset(train_data, train_audio)
val_ds   = SepDataset(val_data, val_audio)
train_loader = DataLoader(train_ds, batch_size=SEP_CFG['batch_size'],
                           shuffle=True, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=SEP_CFG['batch_size'],
                           shuffle=False, num_workers=0)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# ═══════════════════════════════════════════════════════════════════════
# 3.4 Архитектура
# ═══════════════════════════════════════════════════════════════════════
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1),
            nn.BatchNorm2d(cout), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1),
            nn.BatchNorm2d(cout), nn.LeakyReLU(0.1, inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class UNetCore(nn.Module):
    """Общий энкодер-декодер. Возвращает признаки перед выходом: (B, base, F, T)."""
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.pool = nn.MaxPool2d(2)
        self.e1 = ConvBlock(1,   b);   self.e2 = ConvBlock(b,   b*2)
        self.e3 = ConvBlock(b*2, b*4); self.e4 = ConvBlock(b*4, b*8)
        self.bot = ConvBlock(b*8, b*16)
        self.d4 = ConvBlock(b*16 + b*8, b*8)
        self.d3 = ConvBlock(b*8 + b*4, b*4)
        self.d2 = ConvBlock(b*4 + b*2, b*2)
        self.d1 = ConvBlock(b*2 + b, b)

    def forward(self, x):
        e1 = self.e1(x)
        e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2))
        e4 = self.e4(self.pool(e3))
        z  = self.bot(self.pool(e4))
        d4 = self.d4(torch.cat([F.interpolate(z,  e4.shape[-2:], mode='bilinear', align_corners=False), e4], 1))
        d3 = self.d3(torch.cat([F.interpolate(d4, e3.shape[-2:], mode='bilinear', align_corners=False), e3], 1))
        d2 = self.d2(torch.cat([F.interpolate(d3, e2.shape[-2:], mode='bilinear', align_corners=False), e2], 1))
        d1 = self.d1(torch.cat([F.interpolate(d2, e1.shape[-2:], mode='bilinear', align_corners=False), e1], 1))
        return d1

class SeparationMagnitude(nn.Module):
    """Softmax magnitude masks: sum по источникам = 1."""
    def __init__(self, base=32, s_max=S_MAX):
        super().__init__()
        self.core = UNetCore(base)
        self.out = nn.Conv2d(base, s_max, 1)
    def forward(self, x):
        return F.softmax(self.out(self.core(x)), dim=1)

class SeparationComplex(nn.Module):
    """Complex masks (cIRM): real + imag через tanh в [-1, 1]."""
    def __init__(self, base=32, s_max=S_MAX):
        super().__init__()
        self.core = UNetCore(base)
        self.out = nn.Conv2d(base, s_max * 2, 1)
        self.s_max = s_max
    def forward(self, x):
        return torch.tanh(self.out(self.core(x)))

# ═══════════════════════════════════════════════════════════════════════
# 3.5 Forward pass
# ═══════════════════════════════════════════════════════════════════════
def separate_batch(model, mix):
    """Возвращает (est_wave, est_spec, masks, mix_spec)."""
    mix_spec = stft(mix)
    mix_mag = mix_spec.abs()

    if MODE == "magnitude_sdr":
        masks = model(mix_mag.unsqueeze(1))               # (B, S, F, T) softmax
        est_spec = masks * mix_spec.unsqueeze(1)
    else:  # complex
        masks_raw = model(mix_mag.unsqueeze(1))           # (B, 2S, F, T) tanh
        s = S_MAX
        mask_complex = torch.complex(masks_raw[:, :s], masks_raw[:, s:])
        masks = masks_raw
        est_spec = mask_complex * mix_spec.unsqueeze(1)

    B, S = est_spec.shape[:2]
    est_wave = istft(est_spec.reshape(B * S, F_BINS, -1),
                     length=mix.shape[-1]).reshape(B, S, -1)
    return est_wave, est_spec, masks, mix_spec

# ═══════════════════════════════════════════════════════════════════════
# 3.6 SI-SDR
# ═══════════════════════════════════════════════════════════════════════
def si_sdr(pred, target, eps=1e-8):
    pred   = pred   - pred.mean(-1, keepdim=True)
    target = target - target.mean(-1, keepdim=True)
    alpha = (pred * target).sum(-1, keepdim=True) / (target.pow(2).sum(-1, keepdim=True) + eps)
    ts = alpha * target
    num = ts.pow(2).sum(-1)
    den = (pred - ts).pow(2).sum(-1) + eps
    return 10 * torch.log10(num / den + eps)

# ═══════════════════════════════════════════════════════════════════════
# 3.7 PIT loss (L1_magnitude + опционально -SI-SDR)
# ═══════════════════════════════════════════════════════════════════════
def pit_loss(est_wave, est_spec, tgt_wave, tgt_spec, n_sources,
             sdr_weight=0.0, unmatched_w=0.3):
    """
    est_wave, tgt_wave:  (B, S_max, T)
    est_spec, tgt_spec:  (B, S_max, F, T) complex
    """
    B, S = est_wave.shape[:2]
    losses = []

    for b in range(B):
        n = int(n_sources[b])
        est_mag_b = est_spec[b].abs()               # (S, F, T)
        tgt_mag_b = tgt_spec[b, :n].abs()           # (n, F, T)

        # L1 cost
        l1_cost = torch.zeros(S, n, device=est_wave.device)
        for i in range(S):
            for j in range(n):
                l1_cost[i, j] = F.l1_loss(est_mag_b[i], tgt_mag_b[j])

        # SI-SDR (только если используется)
        if sdr_weight > 0:
            sdr_mat = si_sdr(
                est_wave[b].unsqueeze(1),          # (S, 1, T)
                tgt_wave[b, :n].unsqueeze(0),      # (1, n, T)
            )                                       # → (S, n)
            combined = l1_cost - sdr_weight * sdr_mat
        else:
            sdr_mat = None
            combined = l1_cost

        rows, cols = linear_sum_assignment(combined.detach().cpu().numpy())

        matched = set()
        loss_b = torch.zeros((), device=est_wave.device)
        for r, c in zip(rows, cols):
            if sdr_weight > 0:
                loss_b = loss_b + (1 - sdr_weight) * l1_cost[r, c] \
                                  + sdr_weight * (-sdr_mat[r, c])
            else:
                loss_b = loss_b + l1_cost[r, c]
            matched.add(r)

        for i in range(S):
            if i not in matched:
                loss_b = loss_b + unmatched_w * est_wave[b, i].abs().mean()

        losses.append(loss_b / max(n, 1))

    return torch.stack(losses).mean()

# ═══════════════════════════════════════════════════════════════════════
# 3.8 Train/eval steps
# ═══════════════════════════════════════════════════════════════════════
def train_epoch_sep(model, loader, opt):
    model.train()
    total, n = 0.0, 0
    pbar = tqdm(loader, desc="train-sep", leave=False)
    for batch in pbar:
        mix = batch['mix'].to(DEVICE)
        src = batch['sources'].to(DEVICE)
        nsrc = batch['n_sources'].to(DEVICE)

        opt.zero_grad(set_to_none=True)
        est_wave, est_spec, _, _ = separate_batch(model, mix)

        B = mix.shape[0]
        tgt_spec = stft(src.reshape(B * S_MAX, -1)).reshape(B, S_MAX, F_BINS, -1)

        loss = pit_loss(
            est_wave, est_spec, src, tgt_spec, nsrc,
            sdr_weight=SEP_CFG['sdr_loss_weight'] if MODE == "magnitude_sdr" else 0.0,
            unmatched_w=SEP_CFG['unmatched_loss_weight'],
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), SEP_CFG['gradient_clip'])
        opt.step()

        total += loss.item(); n += 1
        if n % 10 == 0:
            pbar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{total/n:.4f}")
    return total / max(1, n)

@torch.no_grad()
def eval_epoch_sep(model, loader):
    model.eval()
    total, n = 0.0, 0
    sdr_matched, sdr_baseline = [], []
    pbar = tqdm(loader, desc="val-sep", leave=False)
    for batch in pbar:
        mix = batch['mix'].to(DEVICE)
        src = batch['sources'].to(DEVICE)
        nsrc = batch['n_sources'].to(DEVICE)

        est_wave, est_spec, _, _ = separate_batch(model, mix)

        B = mix.shape[0]
        tgt_spec = stft(src.reshape(B * S_MAX, -1)).reshape(B, S_MAX, F_BINS, -1)

        loss = pit_loss(
            est_wave, est_spec, src, tgt_spec, nsrc,
            sdr_weight=SEP_CFG['sdr_loss_weight'] if MODE == "magnitude_sdr" else 0.0,
            unmatched_w=SEP_CFG['unmatched_loss_weight'],
        )
        total += loss.item(); n += 1

        # per-sample SDR
        for b in range(B):
            k = int(nsrc[b])
            sdr_mat = si_sdr(
                est_wave[b].unsqueeze(1),          # (S, 1, T)
                src[b, :k].unsqueeze(0),           # (1, k, T)
            )                                       # → (S, k)
            rows, cols = linear_sum_assignment(-sdr_mat.cpu().numpy())
            sdr_matched.extend([sdr_mat[r, c].item() for r, c in zip(rows, cols)])
            baseline = si_sdr(mix[b:b+1].expand(k, -1), src[b, :k])
            sdr_baseline.extend(baseline.cpu().tolist())

    mean_sdr = float(np.mean(sdr_matched))
    mean_base = float(np.mean(sdr_baseline))
    return dict(
        loss=total / max(1, n),
        matched_sdr=mean_sdr,
        baseline_sdr=mean_base,
        si_sdri=mean_sdr - mean_base,
    )

# ═══════════════════════════════════════════════════════════════════════
# 3.9 Инициализация модели + optimizer
# ═══════════════════════════════════════════════════════════════════════
if MODE == "magnitude_sdr":
    sep_model = SeparationMagnitude(base=SEP_CFG['base'], s_max=S_MAX).to(DEVICE)
else:
    sep_model = SeparationComplex(base=SEP_CFG['base'], s_max=S_MAX).to(DEVICE)

n_params = sum(p.numel() for p in sep_model.parameters())
print(f"\n{MODE} U-Net: {n_params/1e6:.2f}M params")

opt_sep = torch.optim.AdamW(sep_model.parameters(),
                            lr=SEP_CFG['lr'], weight_decay=SEP_CFG['wd'])
sched_sep = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt_sep, T_max=SEP_CFG['epochs'], eta_min=1e-6)

# ═══════════════════════════════════════════════════════════════════════
# 3.10 Чекпоинты (не перезаписываем best_v3.pt!)
# ═══════════════════════════════════════════════════════════════════════
CKPT_SEP      = CKPT / "separation" / f"best_v{SEP_VERSION}_{MODE}.pt"
CKPT_SEP_LAST = CKPT / "separation" / f"last_v{SEP_VERSION}_{MODE}.pt"
CKPT_BASELINE = CKPT / "separation" / "best_v3.pt"

print(f"New best: {CKPT_SEP.name}")
print(f"New last: {CKPT_SEP_LAST.name}")
if CKPT_BASELINE.exists():
    print(f"Baseline v3 preserved: {CKPT_BASELINE.name} (+1.61 dB)")

# ═══════════════════════════════════════════════════════════════════════
# 3.11 Цикл обучения
# ═══════════════════════════════════════════════════════════════════════
RESUME_SEP = False   # ← True для продолжения с last

start_epoch = 1
best_sdr = -999.0
best_state_sep = None
wait = 0
history_sep = []

if RESUME_SEP and CKPT_SEP_LAST.exists():
    print("\nLoading last checkpoint for resume...")
    last = torch.load(CKPT_SEP_LAST, map_location=DEVICE, weights_only=False)
    if last.get('version') != SEP_VERSION or last.get('mode') != MODE:
        raise RuntimeError(f"Version/mode mismatch: "
                           f"{last.get('version')}/{last.get('mode')} "
                           f"≠ {SEP_VERSION}/{MODE}")
    sep_model.load_state_dict(last['model_state'])
    opt_sep.load_state_dict(last['optimizer_state'])
    sched_sep.load_state_dict(last['scheduler_state'])
    start_epoch = last['epoch'] + 1
    best_sdr = last['best_sdr']
    wait = last['wait']
    history_sep = last['history']
    if CKPT_SEP.exists():
        best_state_sep = torch.load(CKPT_SEP, map_location=DEVICE,
                                     weights_only=False)['model_state']
    print(f"Resumed from E{start_epoch}, best={best_sdr:+.2f} dB")

if DO_TRAIN:
    print("\n" + "=" * 65)
    print(f"STARTING SEPARATION TRAINING — MODE={MODE}")
    print("=" * 65)

    for epoch in range(start_epoch, SEP_CFG['epochs'] + 1):
        t0 = time.time()
        tr = train_epoch_sep(sep_model, train_loader, opt_sep)
        va = eval_epoch_sep(sep_model, val_loader)
        sched_sep.step()

        history_sep.append(dict(epoch=epoch, train_loss=tr, **va))
        print(f"E{epoch:02d} | tr={tr:.4f} val={va['loss']:.4f} | "
              f"SDR={va['matched_sdr']:+.2f} (base {va['baseline_sdr']:+.2f}) | "
              f"SI-SDRi={va['si_sdri']:+.2f} | {time.time()-t0:.0f}s")

        if va['si_sdri'] > best_sdr:
            best_sdr = va['si_sdri']
            best_state_sep = {k: v.clone() for k, v in sep_model.state_dict().items()}
            wait = 0
            atomic_torch_save(
                dict(version=SEP_VERSION, mode=MODE,
                     model_state=best_state_sep,
                     history=history_sep, best_si_sdri=best_sdr,
                     config=SEP_CFG, epoch=epoch),
                CKPT_SEP,
            )
            print(f"  → best {best_sdr:+.2f} dB")
        else:
            wait += 1

        atomic_torch_save(
            dict(version=SEP_VERSION, mode=MODE,
                 model_state=sep_model.state_dict(),
                 optimizer_state=opt_sep.state_dict(),
                 scheduler_state=sched_sep.state_dict(),
                 epoch=epoch, best_sdr=best_sdr, wait=wait,
                 history=history_sep),
            CKPT_SEP_LAST,
        )

        if wait >= SEP_CFG['patience']:
            print(f"Early stop at E{epoch}")
            break

    if best_state_sep is not None:
        sep_model.load_state_dict(best_state_sep)

    print(f"\n{'='*65}")
    print(f"Separation [{MODE}] done. Best SI-SDRi = {best_sdr:+.2f} dB")
    if CKPT_BASELINE.exists():
        v3 = torch.load(CKPT_BASELINE, map_location='cpu', weights_only=False)
        v3_sdr = v3['best_si_sdri']
        print(f"Baseline v3:     {v3_sdr:+.2f} dB")
        print(f"Δ vs v3:         {best_sdr - v3_sdr:+.2f} dB")
    print(f"Best checkpoint: {CKPT_SEP}")
    print(f"{'='*65}")

else:
    # CPU / build-only режим
    print("\n" + "=" * 65)
    print("DO_TRAIN=False — обучение пропущено")
    print("=" * 65)
    print(f"Модель построена: {n_params/1e6:.2f}M params")
    print(f"MODE = {MODE}")
    print(f"LR = {SEP_CFG['lr']}")
    if MODE == "magnitude_sdr":
        print(f"Loss: (1-{SEP_CFG['sdr_loss_weight']})·L1 + "
              f"{SEP_CFG['sdr_loss_weight']}·(-SI-SDR) через PIT")

    # Проверим, что forward-pass работает (без обучения)
    print("\nSmoke-test (forward pass на 1 батче, без градиентов)...")
    sep_model.eval()
    batch = next(iter(val_loader))
    mix = batch['mix'][:2].to(DEVICE)
    with torch.no_grad():
        est_wave, est_spec, masks, mix_spec = separate_batch(sep_model, mix)
    print(f"  mix:      {tuple(mix.shape)}")
    print(f"  est_wave: {tuple(est_wave.shape)}")
    print(f"  est_spec: {tuple(est_spec.shape)}  ({est_spec.dtype})")
    print(f"  masks:    {tuple(masks.shape)}")

    if MODE == "magnitude_sdr":
        mask_sum = masks.sum(dim=1)
        print(f"  mask sum: mean={mask_sum.mean():.4f} std={mask_sum.std():.4f} "
              f"(должно быть 1.000/0.000)")
    else:
        print(f"  masks range: [{masks.min():.3f}, {masks.max():.3f}] "
              f"(tanh → [-1, 1])")

    # Сравнение с baseline
    if CKPT_BASELINE.exists():
        v3 = torch.load(CKPT_BASELINE, map_location='cpu', weights_only=False)
        print(f"\nBaseline v3 (не тронут): SI-SDRi = {v3['best_si_sdri']:+.2f} dB")
        print("После запуска на GPU проверим, побьёт ли новый режим этот baseline.")

    print("\n" + "=" * 65)
    print("Готово к запуску на GPU")
    print("=" * 65)
    print("Когда GPU вернётся:")
    print("  1. Перезапустить runtime, T4 GPU")
    print("  2. Запустить Module 1 (мгновенно — всё на Drive)")
    print("  3. Запустить Module 3 с DO_TRAIN=True")
    print(f"  4. Ожидаемое время: ~1 ч на {MODE}")
    print(f"  5. Ожидаемый SI-SDRi: "
          f"{'+2.5…+4 dB' if MODE == 'magnitude_sdr' else '+3…+8 dB'}")

Bootstrap: не нужен — зависимости уже в памяти.
MODE = magnitude_sdr
SEP_CFG = {'batch_size': 8, 'base': 32, 'epochs': 25, 'lr': 0.0003, 'wd': 1e-05, 'patience': 6, 'gradient_clip': 5.0, 'sdr_loss_weight': 0.5, 'unmatched_loss_weight': 0.3}

Train batches: 400, Val batches: 50

magnitude_sdr U-Net: 7.85M params
New best: best_v4_magnitude_sdr.pt
New last: last_v4_magnitude_sdr.pt
Baseline v3 preserved: best_v3.pt (+1.61 dB)

STARTING SEPARATION TRAINING — MODE=magnitude_sdr


train-sep:   0%|          | 0/400 [00:00<?, ?it/s]

val-sep:   0%|          | 0/50 [00:00<?, ?it/s]

E01 | tr=1.3610 val=1.1480 | SDR=-2.92 (base -5.78) | SI-SDRi=+2.87 | 352s
  → best +2.87 dB


train-sep:   0%|          | 0/400 [00:00<?, ?it/s]

val-sep:   0%|          | 0/50 [00:00<?, ?it/s]

... (58 outputs left out)

train-sep:   0%|          | 0/400 [00:00<?, ?it/s]

val-sep:   0%|          | 0/50 [00:00<?, ?it/s]

E22 | tr=-0.5842 val=0.2108 | SDR=-1.21 (base -5.78) | SI-SDRi=+4.57 | 317s
  → best +4.57 dB


train-sep:   0%|          | 0/400 [00:00<?, ?it/s]

In [1]:
# ═══════════════════════════════════════════════════════════════════════
# MODULE 4 — INTEGRATION: SED + SEPARATION
# v1: source-level accuracy через PIT matching
# Работает на CPU (можно запускать без GPU)
# ═══════════════════════════════════════════════════════════════════════
import os, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import librosa, soundfile as sf
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from scipy.optimize import linear_sum_assignment
from scipy.ndimage import median_filter, maximum_filter1d
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════
# 4.0 BOOTSTRAP (auto — как в Module 3)
# ═══════════════════════════════════════════════════════════════════════
try:
    _ = (train_data, val_data, train_audio, val_audio,
         N_SAMPLES, N_FFT, HOP, S_MAX, CACHE, CKPT, DEVICE,
         atomic_torch_save, reconstruct_sources, class_names, val_meta,
         SR, N_FRAMES, N_CLASSES, F_BINS)
    print("Bootstrap: не нужен — зависимости уже в памяти.")
except NameError as e:
    print(f"Auto-bootstrap: '{e.name}' не найден. Загружаю с Drive (~15 сек)...")
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    ROOT = Path("/content/drive/MyDrive/ESC50_Sound_Separation")
    DATA_DIR = ROOT / "data" / "ESC-50"
    CACHE = ROOT / "cache"
    CKPT = ROOT / "checkpoints"
    OUT = ROOT / "outputs"
    OUT.mkdir(parents=True, exist_ok=True)

    SR, DURATION = 22050, 4.0
    N_SAMPLES = int(SR * DURATION)
    N_FFT, HOP, N_MELS = 1024, 256, 128
    N_FRAMES = 1 + N_SAMPLES // HOP
    N_CLASSES, S_MAX = 50, 3
    F_BINS = N_FFT // 2 + 1
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"  DEVICE = {DEVICE}")

    def atomic_torch_save(state, path):
        path = Path(path); tmp = path.parent / (path.name + ".tmp")
        torch.save(state, tmp); os.replace(tmp, path)

    metadata = pd.read_csv(DATA_DIR / "meta" / "esc50.csv").sort_values(
        ["fold", "filename"]).reset_index(drop=True)
    class_names = metadata[["target", "category"]].drop_duplicates(
        ).sort_values("target")["category"].tolist()
    val_meta = metadata[metadata["fold"] == 5].reset_index(drop=True)

    def _load_split(split):
        d = CACHE / f"{split}_mix_npy"
        keys = ["mix", "frame_labels", "clip_labels", "n_sources", "scale",
                "source_idx", "source_crop_start", "source_start",
                "source_length", "source_gain"]
        return {k: np.load(d / f"{k}.npy", mmap_mode='r') for k in keys}

    train_data = _load_split("train")
    val_data = _load_split("val")
    train_audio = np.load(CACHE / "train_audio.npy", mmap_mode='r')
    val_audio = np.load(CACHE / "val_audio.npy", mmap_mode='r')

    def reconstruct_sources(data_dict, audio_cache, idx, s_max=S_MAX):
        n = int(data_dict['n_sources'][idx])
        scale = float(data_dict['scale'][idx])
        sources = np.zeros((s_max, N_SAMPLES), dtype=np.float32)
        for k in range(n):
            a_idx = int(data_dict['source_idx'][idx, k])
            c_start = int(data_dict['source_crop_start'][idx, k])
            s_start = int(data_dict['source_start'][idx, k])
            s_len = int(data_dict['source_length'][idx, k])
            gain = float(data_dict['source_gain'][idx, k])
            crop = audio_cache[a_idx][c_start:c_start + s_len] * gain * scale
            end = min(s_start + s_len, N_SAMPLES)
            sources[k, s_start:end] = crop[:end - s_start]
        return sources, n

    print("✅ Auto-bootstrap завершён.")

# ═══════════════════════════════════════════════════════════════════════
# 4.1 Конфигурация
# ═══════════════════════════════════════════════════════════════════════
PANNS_CKPT = ROOT / "panns_data" / "Cnn14_DecisionLevelMax.pth"
SED_CKPT   = CKPT / "sed" / "best_2048_v4.pt"
SEP_CKPT   = CKPT / "separation" / "best_v4_magnitude_sdr.pt"
FEAT_VAL   = CACHE / "panns_feats_2048" / "val.npy"

for p in [PANNS_CKPT, SED_CKPT, SEP_CKPT, FEAT_VAL]:
    assert p.exists(), f"Не найден: {p}"

OUT_DIR = OUT / "module4_integration"
OUT_DIR.mkdir(parents=True, exist_ok=True)
WAV_DIR = OUT_DIR / "audio_examples"
WAV_DIR.mkdir(parents=True, exist_ok=True)

# Параметры
N_VAL_EVAL = 400        # сколько val-сэмплов оценивать (400 = все; можно 100 для скорости)
SED_THRESHOLD = 0.5     # порог для clip-F1
SMOOTH_KERNEL = 5       # медианное сглаживание для event F1
PANNS_BATCH = 8         # батч для PANNs inference (CPU: 4-8)
SEP_BATCH = 8           # батч для separation

print(f"SED checkpoint:  {SED_CKPT.name}")
print(f"Sep checkpoint:  {SEP_CKPT.name}")
print(f"Val samples:     {N_VAL_EVAL}")
print(f"Output:          {OUT_DIR}")

# ═══════════════════════════════════════════════════════════════════════
# 4.2 Загрузка SED (BiGRUHead 2048-dim)
# ═══════════════════════════════════════════════════════════════════════
class BiGRUHead(nn.Module):
    def __init__(self, in_dim=2048, num_classes=N_CLASSES, hidden=128, layers=1, dropout=0.5):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout))
        self.gru = nn.GRU(hidden, hidden, num_layers=layers, batch_first=True,
                          bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.frame_head = nn.Linear(hidden * 2, num_classes)
        self.clip_head = nn.Sequential(
            nn.Linear(hidden * 2, 128), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        x = self.proj(x)
        h, _ = self.gru(x); h = self.dropout(h)
        return self.frame_head(h).transpose(1, 2), self.clip_head(h.amax(dim=1))

sed_ckpt = torch.load(SED_CKPT, map_location=DEVICE, weights_only=False)
sed_model = BiGRUHead(in_dim=2048,
                      hidden=sed_ckpt['config']['hidden'],
                      layers=sed_ckpt['config']['layers'],
                      dropout=sed_ckpt['config']['dropout']).to(DEVICE)
sed_model.load_state_dict(sed_ckpt['model_state'])
sed_model.eval()
print(f"\nSED loaded: E{sed_ckpt['epoch']}, event_f1_tuned={sed_ckpt['best_event_f1_tuned']:.4f}")

# ═══════════════════════════════════════════════════════════════════════
# 4.3 Загрузка Separation (U-Net magnitude_sdr)
# ═══════════════════════════════════════════════════════════════════════
class ConvBlock(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.LeakyReLU(0.1, inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.LeakyReLU(0.1, inplace=True))
    def forward(self, x): return self.net(x)

class UNetCore(nn.Module):
    def __init__(self, base=32):
        super().__init__()
        b = base
        self.pool = nn.MaxPool2d(2)
        self.e1 = ConvBlock(1, b); self.e2 = ConvBlock(b, b*2)
        self.e3 = ConvBlock(b*2, b*4); self.e4 = ConvBlock(b*4, b*8)
        self.bot = ConvBlock(b*8, b*16)
        self.d4 = ConvBlock(b*16 + b*8, b*8)
        self.d3 = ConvBlock(b*8 + b*4, b*4)
        self.d2 = ConvBlock(b*4 + b*2, b*2)
        self.d1 = ConvBlock(b*2 + b, b)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(self.pool(e1))
        e3 = self.e3(self.pool(e2)); e4 = self.e4(self.pool(e3))
        z = self.bot(self.pool(e4))
        d4 = self.d4(torch.cat([F.interpolate(z, e4.shape[-2:], mode='bilinear', align_corners=False), e4], 1))
        d3 = self.d3(torch.cat([F.interpolate(d4, e3.shape[-2:], mode='bilinear', align_corners=False), e3], 1))
        d2 = self.d2(torch.cat([F.interpolate(d3, e2.shape[-2:], mode='bilinear', align_corners=False), e2], 1))
        d1 = self.d1(torch.cat([F.interpolate(d2, e1.shape[-2:], mode='bilinear', align_corners=False), e1], 1))
        return d1

class SeparationMagnitude(nn.Module):
    def __init__(self, base=32, s_max=S_MAX):
        super().__init__()
        self.core = UNetCore(base)
        self.out = nn.Conv2d(base, s_max, 1)
    def forward(self, x):
        return F.softmax(self.out(self.core(x)), dim=1)

_HANN = torch.hann_window(N_FFT).to(DEVICE)
def stft(x):
    return torch.stft(x, n_fft=N_FFT, hop_length=HOP, window=_HANN, center=True, return_complex=True)
def istft(spec, length=N_SAMPLES):
    return torch.istft(spec, n_fft=N_FFT, hop_length=HOP, window=_HANN, center=True, length=length)

def separate_batch(model, mix):
    mix_spec = stft(mix)
    masks = model(mix_spec.abs().unsqueeze(1))
    est_spec = masks * mix_spec.unsqueeze(1)
    B, S = est_spec.shape[:2]
    est = istft(est_spec.reshape(B*S, *est_spec.shape[2:]), length=mix.shape[-1])
    return est.reshape(B, S, -1), masks, mix_spec

sep_ckpt = torch.load(SEP_CKPT, map_location=DEVICE, weights_only=False)
sep_model = SeparationMagnitude(base=sep_ckpt['config']['base'], s_max=S_MAX).to(DEVICE)
sep_model.load_state_dict(sep_ckpt['model_state'])
sep_model.eval()
print(f"Sep loaded: E{sep_ckpt['epoch']}, SI-SDRi={sep_ckpt['best_si_sdri']:+.2f} dB")

# ═══════════════════════════════════════════════════════════════════════
# 4.4 PANNs hook для извлечения 2048-dim (для separated sources)
# ═══════════════════════════════════════════════════════════════════════
try:
    from panns_inference import SoundEventDetection
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "panns-inference"])
    from panns_inference import SoundEventDetection

print("\nLoading PANNs...")
panns = SoundEventDetection(checkpoint_path=str(PANNS_CKPT), device=str(DEVICE))
panns.model.eval()

def find_target_layer(model):
    candidates = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d) and module.out_channels == 2048:
            candidates.append((name, module))
    if candidates:
        return candidates[-1]
    for name, module in model.named_modules():
        if 'conv_block6' in name.lower():
            return name, module
    raise RuntimeError("Layer 2048 not found")

target_name, target_module = find_target_layer(panns.model)
print(f"Hook: {target_name}")

_captured = {}
def _hook_fn(m, i, o): _captured['feat'] = o
_handle = target_module.register_forward_hook(_hook_fn)

def extract_2048_features(waveforms_np):
    """
    waveforms_np: list[np.ndarray] | (B, N_SAMPLES) at SR=22050
    Возвращает: torch.Tensor (B, N_FRAMES, 2048) на CPU
    """
    if isinstance(waveforms_np, np.ndarray):
        batch = list(waveforms_np)
    else:
        batch = waveforms_np

    prepared = []
    for a in batch:
        a32 = librosa.resample(np.asarray(a, dtype=np.float32),
                                orig_sr=SR, target_sr=32000)
        L = int(4.0 * 32000)
        a32 = np.pad(a32, (0, max(0, L - len(a32))))[:L]
        prepared.append(a32.astype(np.float32))

    with torch.inference_mode():
        _ = panns.inference(np.stack(prepared))

    feat = _captured['feat']                         # (B, 2048, F', T')
    feat = feat.mean(dim=2)                          # (B, 2048, T')
    feat = F.interpolate(feat, size=N_FRAMES, mode='linear', align_corners=False)
    feat = feat.transpose(1, 2).cpu()                # (B, 345, 2048)
    del _captured['feat']
    return feat

# ═══════════════════════════════════════════════════════════════════════
# 4.5 Метрики SED (для валидации на миксе)
# ═══════════════════════════════════════════════════════════════════════
def _smooth(prob, kernel=5):
    if kernel <= 1: return prob
    size = (1,) * (prob.ndim - 1) + (kernel,)
    return median_filter(prob, size=size, mode='nearest')

def _events(b):
    p = np.concatenate([[0], b.astype(np.int8), [0]])
    d = np.diff(p)
    return np.flatnonzero(d == 1), np.flatnonzero(d == -1)

def event_f1_from_prob(prob, target, thr=0.5, iou_thr=0.5):
    pred = prob > thr; tgt = target > 0.5
    tp = fp = fn = 0
    for b in range(pred.shape[0]):
        for k in range(pred.shape[1]):
            ps, pe = _events(pred[b, k]); ts, te = _events(tgt[b, k])
            if len(ts) == 0: fp += len(ps); continue
            if len(ps) == 0: fn += len(ts); continue
            matched = np.zeros(len(ts), dtype=bool)
            for i in range(len(ps)):
                bi, bj = 0.0, -1
                for j in range(len(ts)):
                    if matched[j]: continue
                    inter = max(0, min(pe[i], te[j]) - max(ps[i], ts[j]))
                    union = max(pe[i], te[j]) - min(ps[i], ts[j])
                    iou = inter / (union + 1e-8)
                    if iou > bi: bi, bj = iou, j
                if bi >= iou_thr: matched[bj] = True; tp += 1
                else: fp += 1
            fn += int((~matched).sum())
    p = tp / (tp + fp + 1e-8); r = tp / (tp + fn + 1e-8)
    return 2*p*r/(p+r+1e-8), p, r

def frame_f1(prob, target, thr=0.5, tol=2):
    pred = prob > thr; tgt = target > 0.5
    if tol > 0:
        tgt = maximum_filter1d(tgt.astype(np.float32), size=2*tol+1, axis=-1) > 0.5
    tp = (pred & tgt).sum(); fp = (pred & ~tgt).sum(); fn = (~pred & tgt).sum()
    return float(2*tp / (2*tp + fp + fn + 1e-8))

def clip_map_from_logits(clip_logits, clip_labels):
    prob = torch.sigmoid(clip_logits).cpu().numpy()
    y = clip_labels.cpu().numpy()
    aps = []
    for k in range(y.shape[1]):
        if y[:, k].sum() == 0: continue
        order = np.argsort(-prob[:, k]); ys = y[order, k]
        tp_cum = np.cumsum(ys); prec = tp_cum / (np.arange(len(ys)) + 1)
        aps.append((prec * ys).sum() / ys.sum())
    return float(np.mean(aps)) if aps else 0.0

# ═══════════════════════════════════════════════════════════════════════
# 4.6 SI-SDR
# ═══════════════════════════════════════════════════════════════════════
def si_sdr(pred, target, eps=1e-8):
    pred = pred - pred.mean(-1, keepdim=True)
    target = target - target.mean(-1, keepdim=True)
    alpha = (pred * target).sum(-1, keepdim=True) / (target.pow(2).sum(-1, keepdim=True) + eps)
    ts = alpha * target
    return 10 * torch.log10(ts.pow(2).sum(-1) / ((pred - ts).pow(2).sum(-1) + eps) + eps)

# ═══════════════════════════════════════════════════════════════════════
# 4.7 Загрузка pre-computed features для val_mix
# ═══════════════════════════════════════════════════════════════════════
X_val = np.load(FEAT_VAL, mmap_mode='r')          # (400, 345, 2048) для val_mix
print(f"Val features (mixture): {X_val.shape}")

# ═══════════════════════════════════════════════════════════════════════
# 4.8 Основной цикл интеграции
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print(f"INTEGRATION EVALUATION on {N_VAL_EVAL} val samples")
print(f"{'='*65}")

# Результаты
all_mix_clip_probs = np.zeros((N_VAL_EVAL, N_CLASSES), dtype=np.float32)
all_mix_frame_probs = np.zeros((N_VAL_EVAL, N_CLASSES, N_FRAMES), dtype=np.float32)
all_si_sdri = []
source_level_correct_top1 = []   # boolean, для каждой (est, gt) пары
source_level_correct_top5 = []
per_sample_detail = []

t_start = time.time()

# SED на миксе (батчами, через pre-computed features)
print("\n[1/3] SED на валидационных миксах...")
with torch.no_grad():
    for start in tqdm(range(0, N_VAL_EVAL, 32), desc="SED mixture"):
        end = min(start + 32, N_VAL_EVAL)
        xb = torch.from_numpy(np.asarray(X_val[start:end], dtype=np.float32))
        fl, cl = sed_model(xb)
        all_mix_frame_probs[start:end] = torch.sigmoid(fl).numpy()
        all_mix_clip_probs[start:end] = torch.sigmoid(cl).numpy()

# Separation + SED на источниках
print("\n[2/3] Separation + SED на разделённых источниках...")
with torch.no_grad():
    for start in tqdm(range(0, N_VAL_EVAL, SEP_BATCH), desc="sep+SED"):
        end = min(start + SEP_BATCH, N_VAL_EVAL)
        B = end - start

        # Готовим миксы и GT источники
        mix_batch = np.asarray(val_data['mix'][start:end], dtype=np.float32)
        src_batch = np.zeros((B, S_MAX, N_SAMPLES), dtype=np.float32)
        n_src_batch = []
        gt_source_classes = []   # list of list — классы GT-источников для каждого сэмпла

        for i in range(B):
            idx = start + i
            src, n = reconstruct_sources(val_data, val_audio, idx)
            src_batch[i] = src
            n_src_batch.append(n)
            # Классы GT-источников
            classes_i = [int(val_meta.iloc[int(val_data['source_idx'][idx, k])]['target'])
                         for k in range(n)]
            gt_source_classes.append(classes_i)

        # Separation
        mix_t = torch.from_numpy(mix_batch).to(DEVICE)
        est_wave, _, _ = separate_batch(sep_model, mix_t)   # (B, S_MAX, N_SAMPLES)

        # SI-SDR и PIT matching
        est_wave_cpu = est_wave.cpu()
        src_batch_t = torch.from_numpy(src_batch)

        for i in range(B):
            k = n_src_batch[i]
            # pairwise SI-SDR: (S_MAX, k)
            sdr_mat = si_sdr(est_wave_cpu[i].unsqueeze(1), src_batch_t[i, :k].unsqueeze(0))
            rows, cols = linear_sum_assignment(-sdr_mat.numpy())
            # matched SDR
            mean_sdr = float(np.mean([sdr_mat[r, c].item() for r, c in zip(rows, cols)]))
            # baseline
            base = si_sdr(mix_t[i:i+1].expand(k, -1).cpu(), src_batch_t[i, :k])
            mean_base = float(base.mean().item())
            all_si_sdri.append(mean_sdr - mean_base)

            # Сохраняем assignment для source-level accuracy
            # rows[j] — индекс est-источника, cols[j] — индекс GT-источника
            for r, c in zip(rows, cols):
                gt_cls = gt_source_classes[i][c]     # GT класс
                # SED на estimate[r]:
                est_audio = est_wave_cpu[i, r].numpy()   # (88200,)
                est_feat = extract_2048_features([est_audio]).to(DEVICE)
                _, est_clip_logit = sed_model(est_feat)
                est_probs = torch.sigmoid(est_clip_logit).cpu().numpy()[0]

                # Top-1, top-5
                top1 = int(np.argmax(est_probs))
                top5 = np.argsort(-est_probs)[:5]
                source_level_correct_top1.append(top1 == gt_cls)
                source_level_correct_top5.append(gt_cls in top5)

                per_sample_detail.append({
                    "sample_idx": int(start + i),
                    "est_idx": int(r),
                    "gt_idx": int(c),
                    "gt_class": gt_cls,
                    "gt_class_name": class_names[gt_cls],
                    "top1_class": top1,
                    "top1_class_name": class_names[top1],
                    "top1_prob": float(est_probs[top1]),
                    "correct_top1": top1 == gt_cls,
                    "correct_top5": gt_cls in top5,
                    "si_sdr_matched": float(sdr_mat[r, c].item()),
                })

            # Сохраняем GT источники для визуализации (только для worst/best)
            # Обойдёмся — их можно восстановить по sample_idx

print(f"\n[3/3] Итого: {len(all_si_sdri)} сэмплов, "
      f"{len(source_level_correct_top1)} пар (est, gt)")
print(f"Время: {time.time()-t_start:.1f}s")

# ═══════════════════════════════════════════════════════════════════════
# 4.9 Метрики
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print(f"RESULTS")
print(f"{'='*65}")

# SED на миксе
Yf = np.asarray(val_data['frame_labels'][:N_VAL_EVAL])
Yc = np.asarray(val_data['clip_labels'][:N_VAL_EVAL])

mix_frame_sm = _smooth(all_mix_frame_probs, SMOOTH_KERNEL)
frame_f1_mix = frame_f1(mix_frame_sm, Yf, thr=0.5)
event_f1_mix, _, _ = event_f1_from_prob(mix_frame_sm, Yf, thr=0.5)
best_ev, best_thr = 0.0, 0.5
for t in np.arange(0.1, 0.91, 0.1):
    f1, _, _ = event_f1_from_prob(mix_frame_sm, Yf, thr=float(t))
    if f1 > best_ev: best_ev, best_thr = f1, float(t)
mAP_mix = clip_map_from_logits(
    torch.from_numpy(np.log(all_mix_clip_probs / (1 - all_mix_clip_probs + 1e-8))),
    torch.from_numpy(Yc))

# Separation
mean_si_sdri = float(np.mean(all_si_sdri))

# Source-level
acc_top1 = float(np.mean(source_level_correct_top1))
acc_top5 = float(np.mean(source_level_correct_top5))

print(f"\n[SED на миксе — baseline]")
print(f"  Clip mAP:          {mAP_mix:.4f}")
print(f"  Frame F1 @ 0.5:    {frame_f1_mix:.4f}")
print(f"  Event F1 @ 0.5:    {event_f1_mix:.4f}")
print(f"  Event F1 tuned:    {best_ev:.4f} @ {best_thr:.2f}")

print(f"\n[Separation]")
print(f"  Mean SI-SDRi:      {mean_si_sdri:+.2f} dB")

print(f"\n[Source-level classification — интеграция]")
print(f"  Top-1 accuracy:    {acc_top1:.4f} ({acc_top1*100:.1f}%)")
print(f"  Top-5 recall:      {acc_top5:.4f} ({acc_top5*100:.1f}%)")
print(f"  Всего пар (est, gt): {len(source_level_correct_top1)}")

# ═══════════════════════════════════════════════════════════════════════
# 4.10 Per-class breakdown
# ═══════════════════════════════════════════════════════════════════════
detail_df = pd.DataFrame(per_sample_detail)
per_class = detail_df.groupby('gt_class_name').agg(
    count=('correct_top1', 'size'),
    acc_top1=('correct_top1', 'mean'),
    acc_top5=('correct_top5', 'mean'),
    mean_si_sdr=('si_sdr_matched', 'mean'),
).sort_values('acc_top1', ascending=False)

per_class.to_csv(OUT_DIR / "per_class_source_accuracy.csv")
print(f"\nPer-class: сохранено в per_class_source_accuracy.csv")
print("\nТоп-5 классов (легко):")
for name, row in per_class.head(5).iterrows():
    print(f"  {name:<22} acc1={row['acc_top1']:.2f} acc5={row['acc_top5']:.2f} n={int(row['count'])}")
print("\nХудшие 5:")
for name, row in per_class.tail(5).iterrows():
    print(f"  {name:<22} acc1={row['acc_top1']:.2f} acc5={row['acc_top5']:.2f} n={int(row['count'])}")

# ═══════════════════════════════════════════════════════════════════════
# 4.11 Графики
# ═══════════════════════════════════════════════════════════════════════
print(f"\nСоздание графиков...")

# Plot 1: SI-SDRi distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(all_si_sdri, bins=30, edgecolor='k', alpha=0.7)
axes[0].axvline(0, color='red', ls='--', lw=1, label='SI-SDRi=0')
axes[0].axvline(mean_si_sdri, color='green', ls=':', lw=2, label=f'mean={mean_si_sdri:+.2f}')
axes[0].set_xlabel('SI-SDRi (dB)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Separation SI-SDRi (n={len(all_si_sdri)})')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Plot 2: source-level accuracy
axes[1].bar(['Top-1', 'Top-5'], [acc_top1, acc_top5], color=['steelblue', 'lightgreen'])
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Accuracy')
axes[1].set_title(f'Source-level classification (n={len(source_level_correct_top1)})')
for i, v in enumerate([acc_top1, acc_top5]):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center')
axes[1].grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / "integration_metrics.png", dpi=150, bbox_inches='tight')
plt.close()

# Plot 3: per-class accuracy
fig, ax = plt.subplots(figsize=(14, 5))
acc_sorted = per_class.sort_values('acc_top1', ascending=False)
colors = ['green' if v >= 0.5 else 'orange' if v >= 0.3 else 'red'
          for v in acc_sorted['acc_top1']]
ax.bar(range(len(acc_sorted)), acc_sorted['acc_top1'], color=colors)
ax.set_xticks(range(len(acc_sorted)))
ax.set_xticklabels(acc_sorted.index, rotation=90, fontsize=7)
ax.axhline(acc_top1, color='black', ls='--', lw=1, label=f'mean={acc_top1:.3f}')
ax.set_ylabel('Top-1 accuracy per class')
ax.set_title('Source-level accuracy per class (only classes present in val)')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(OUT_DIR / "per_class_accuracy.png", dpi=150, bbox_inches='tight')
plt.close()

# Plot 4: пример confusion — top-5 (est, gt) пары, где prediction != gt
wrong = detail_df[~detail_df['correct_top1']].head(10)
print("\nПримеры ошибок (predicted vs gt):")
for _, row in wrong.iterrows():
    print(f"  sample {int(row['sample_idx'])}: "
          f"gt='{row['gt_class_name']}' → pred='{row['top1_class_name']}' "
          f"(p={row['top1_prob']:.2f})")

# ═══════════════════════════════════════════════════════════════════════
# 4.12 WAV-примеры (3 best + 3 worst по SI-SDRi)
# ═══════════════════════════════════════════════════════════════════════
print(f"\nСохранение WAV-примеров...")

si_sdri_arr = np.array(all_si_sdri)
best_idx = np.argsort(-si_sdri_arr)[:3]
worst_idx = np.argsort(si_sdri_arr)[:3]

for tag, indices in [("best", best_idx), ("worst", worst_idx)]:
    for rank, sample_idx in enumerate(indices):
        sample_idx = int(sample_idx)
        # Восстанавливаем аудио
        mix = np.asarray(val_data['mix'][sample_idx], dtype=np.float32)
        src, n = reconstruct_sources(val_data, val_audio, sample_idx)

        # Separation
        with torch.no_grad():
            mix_t = torch.from_numpy(mix[None]).to(DEVICE)
            est, _, _ = separate_batch(sep_model, mix_t)
            est = est[0].cpu().numpy()

        # Сохраняем
        prefix = f"{tag}_{rank}_sample{sample_idx}"
        sf.write(WAV_DIR / f"{prefix}_mix.wav", mix, SR)
        for k in range(n):
            sf.write(WAV_DIR / f"{prefix}_gt_{k}.wav", src[k], SR)
        for j in range(S_MAX):
            sf.write(WAV_DIR / f"{prefix}_est_{j}.wav", est[j], SR)

print(f"WAV-файлы: {WAV_DIR}/  (6 примеров × ~7 файлов)")

# ═══════════════════════════════════════════════════════════════════════
# 4.13 JSON-отчёт
# ═══════════════════════════════════════════════════════════════════════
report = {
    "module": "Module 4 — Integration (SED + Separation)",
    "n_val_samples": int(N_VAL_EVAL),
    "n_source_pairs": len(source_level_correct_top1),
    "sed_on_mixture": {
        "clip_mAP": float(mAP_mix),
        "frame_f1_at_0.5": float(frame_f1_mix),
        "event_f1_at_0.5": float(event_f1_mix),
        "event_f1_tuned": float(best_ev),
        "best_threshold": float(best_thr),
    },
    "separation": {
        "mean_si_sdri_dB": float(mean_si_sdri),
        "std_si_sdri_dB": float(np.std(all_si_sdri)),
    },
    "source_level_classification": {
        "top1_accuracy": float(acc_top1),
        "top5_recall": float(acc_top5),
        "description": (
            "Для каждой пары (разделённый источник, GT источник) после PIT-matching "
            "проверяется, что SED top-1 на разделённом источнике совпадает с GT-классом. "
            "Это метрика того, насколько хорошо separation сохраняет информацию о классе."
        ),
    },
    "checkpoints": {
        "sed": str(SED_CKPT),
        "separation": str(SEP_CKPT),
    },
}
with open(OUT_DIR / "report.json", "w") as f:
    json.dump(report, f, indent=2)

detail_df.to_csv(OUT_DIR / "per_sample_predictions.csv", index=False)

print(f"\n{'='*65}")
print(f"MODULE 4 COMPLETE")
print(f"{'='*65}")
print(f"\nОтчёт: {OUT_DIR / 'report.json'}")
print(f"Графики: {OUT_DIR}/*.png")
print(f"WAV: {WAV_DIR}/")
print(f"Predictions: {OUT_DIR / 'per_sample_predictions.csv'}")

print(f"\n{'='*65}")
print(f"ФИНАЛЬНЫЕ МЕТРИКИ ПАЙПЛАЙНА")
print(f"{'='*65}")
print(f"SED на миксе:")
print(f"  Clip mAP:        {mAP_mix:.4f}")
print(f"  Event F1 tuned:  {best_ev:.4f}")
print(f"\nSeparation:")
print(f"  Mean SI-SDRi:    {mean_si_sdri:+.2f} dB")
print(f"\nSource-level (интеграция):")
print(f"  Top-1 accuracy:  {acc_top1:.4f} ({acc_top1*100:.1f}%)")
print(f"  Top-5 recall:    {acc_top5:.4f}")

Auto-bootstrap: 'train_data' не найден. Загружаю с Drive (~15 сек)...
Mounted at /content/drive
  DEVICE = cpu
✅ Auto-bootstrap завершён.
SED checkpoint:  best_2048_v4.pt
Sep checkpoint:  best_v4_magnitude_sdr.pt
Val samples:     400
Output:          /content/drive/MyDrive/ESC50_Sound_Separation/outputs/module4_integration

SED loaded: E25, event_f1_tuned=0.4199
Sep loaded: E22, SI-SDRi=+4.57 dB

Loading PANNs...
Checkpoint path: /content/drive/MyDrive/ESC50_Sound_Separation/panns_data/Cnn14_DecisionLevelMax.pth
Using CPU.
Hook: conv_block6.conv2
Val features (mixture): (400, 345, 2048)

INTEGRATION EVALUATION on 400 val samples

[1/3] SED на валидационных миксах...


SED mixture:   0%|          | 0/13 [00:00<?, ?it/s]


[2/3] Separation + SED на разделённых источниках...


sep+SED:   0%|          | 0/50 [00:00<?, ?it/s]


[3/3] Итого: 400 сэмплов, 1004 пар (est, gt)
Время: 1638.3s

RESULTS

[SED на миксе — baseline]
  Clip mAP:          0.4330
  Frame F1 @ 0.5:    0.3917
  Event F1 @ 0.5:    0.3437
  Event F1 tuned:    0.4199 @ 0.70

[Separation]
  Mean SI-SDRi:      +4.51 dB

[Source-level classification — интеграция]
  Top-1 accuracy:    0.2321 (23.2%)
  Top-5 recall:      0.4084 (40.8%)
  Всего пар (est, gt): 1004

Per-class: сохранено в per_class_source_accuracy.csv

Топ-5 классов (легко):
  clock_alarm            acc1=0.93 acc5=1.00 n=15
  siren                  acc1=0.88 acc5=0.92 n=24
  pouring_water          acc1=0.67 acc5=0.67 n=18
  rooster                acc1=0.65 acc5=0.75 n=20
  hand_saw               acc1=0.59 acc5=0.65 n=17

Худшие 5:
  cat                    acc1=0.00 acc5=0.14 n=14
  clock_tick             acc1=0.00 acc5=0.00 n=23
  fireworks              acc1=0.00 acc5=0.07 n=15
  mouse_click            acc1=0.00 acc5=0.05 n=22
  wind                   acc1=0.00 acc5=0.06 n=17

Создан